# Trade Area Modeling
According to [Market Business News](https://marketbusinessnews.com/financial-glossary/trade-area-definition-meaning/), a trade (or market) area is the geographical area  where all or most of a business' sales volume occurs. It can be used to make decisions about the optimum location for business growth and expansion.

There are different trade area model and in this use case, we will be using the widely accepted [Huff model](https://en.wikipedia.org/wiki/Huff_model). The Huff model uses the **distance** between customers and existing business locations, or locations that are planned for the future, along with the **attractiveness of those locations**, to determine the likelihood of customers visiting those locations.

## Case Study
SereneGlow is a fictional beauty and personal care brand that has thrived as an exclusively online business for the past two years. They serve a predominantly US market, have done some market research and this is what they have discovered.
1. While the e-commerce share of total retail sales continues to rise, in-store shopping is still the preferred method for most US cusstomers and makes up a very large chunk of total retail sales. [[Reference](https://capitaloneshopping.com/research/online-vs-in-store-shopping-statistics/)]
2. More online shoppers are opting to pick up orders in-store and shoppers are generally returning to stores; retailers are reaching customers with small-format stores to encourage impulse purchases and reach new markets, among other benefits[[Reference](https://www2.deloitte.com/us/en/pages/consulting/articles/q1-2023-consumer-trends-report.html)]
3. Online businesses are developing physical footprints to keep their customers loyal and give them additional touchpoints for the brand; customers themselves now prefer omnichannel experiences [[Reference](https://www.inc.com/rebecca-deczynski/the-future-of-retail-isnt-direct-to-consumer-brands-embracing-brick-mortar-2023.html)]

In light of these, SereneGlow has decided to build their physical presence. To start, they have also decided to leverage partnerships with already existing [top department stores](https://www.junglescout.com/wp-content/uploads/2023/09/Jungle-Scout-Consumer-Trends-Report-Q3-2023.pdf) in the US like Walmart, Target, Kohl's etc. They will be leveraging the small-format store approach by drilling down to neighborhood locations.
They have decided to use a trade area model to know which cities to focus on and which store locations should be their areas of focus. The data for this model has already been mined in a previous notebook.

#### Modules and Clients

In [1]:
# import relevant packages
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

#### Data Loading

In [2]:
# load datasets
customer_communities = pd.read_csv('data/customer_communities.csv')
store_distances = pd.read_csv('data/store_distances.csv')
store_preferences = pd.read_csv('data/store_preferences.csv')
store_data = pd.read_csv('data/store_data.csv')

#### Data Inspection and Cleaning

In [3]:
#inspect customer communities
customer_communities.head()

,location,neighborhoods,population,table_date,sales
0,"Albuquerque, New Mexico",Sun North Estates,93,2023-11-21,597.806375
1,"Albuquerque, New Mexico",Cherry Hills,842,2023-11-21,5412.397501
2,"Albuquerque, New Mexico",Alameda N Valley,5,2023-11-21,32.140128
3,"Albuquerque, New Mexico",West Mesa,4249,2023-11-21,27312.680502
4,"Albuquerque, New Mexico",Jerry Cline Park,226,2023-11-21,1452.733771


In [4]:
# inspect store data
store_data.head(2)

,element_type,osmid,name,address,geometry,building:levels,store_location,city_location
0,relation,6682169,Walmart Supercenter,"8500 Washington Blvd, Pico Rivera","POLYGON ((-118.1050108 33.9856076, -118.105001...",NaN,"Walmart Supercenter, 8500 Washington Blvd, Pic...","Los Angeles, California"
1,relation,7326273,Walmart,"8333 Van Nuys Blvd, Panorama City","POLYGON ((-118.450057 34.2223448, -118.4501609...",NaN,"Walmart, 8333 Van Nuys Blvd, Panorama City","Los Angeles, California"


In [5]:
# filter for sales in communities with store distances
community_sales = customer_communities.loc[(customer_communities['location'].isin(store_data['city_location'])),
                                             ['neighborhoods','qtrly_sales']]

In [6]:
# inspect store distances
store_distances.head()

,neighborhoods,"Walmart Supercenter, 8500 Washington Blvd, Pico Rivera","Walmart, 8333 Van Nuys Blvd, Panorama City","Walmart Supercenter, 7250 Carson Blvd, Long Beach","Walmart Supercenter, 19821 Rinaldi St, Porter Ranch","Target, 8840 Corbin Ave, Northridge","Kohl's, 8800 Corbin Ave, Northridge","Target, 11051 Victory Blvd, North Hollywood","Kohl's, 1201 S Fremont Ave, Alhambra","Target, 5500 W Sunset Blvd, Los Angeles",...,"JCPenney, 332 Stonewood St, Downey","Walmart Supercenter, 14501 Lakewood Blvd, Paramount","Target, 1800 Empire Ave, Burbank","Walmart Supercenter, 1301 N Victory Pl, Burbank","Walmart Supercenter, 26471 Carl Boyer Dr, Santa Clarita","Kohl's, 24200 Valencia Blvd, Valencia","JCPenney, 9301 Tampa Ave, Northridge","Walmart Supercenter, 9001 Apollo Way, Downey","Target, 3433 Sepulveda Blvd, Torrance","Kohl's, 872 New Los Angeles Ave, Moorpark"
0,Highland Park,24.5,34.4,44.2,51.7,55.2,55.5,25.4,6.9,17.9,...,28.8,32.5,20.5,19.3,58.2,60.0,54.5,30.0,42.6,81.3
1,Playa Vista,40.1,33.4,45.8,46.3,41.3,39.5,37.6,34.0,21.2,...,35.4,32.6,42.1,42.1,56.4,70.1,41.1,33.9,23.5,76.1
2,University Hills,16.2,39.2,36.3,52.0,55.4,55.6,29.7,3.9,17.3,...,19.8,23.5,26.0,24.8,68.8,61.8,64.2,24.9,42.8,81.4
3,North Hills,55.8,4.7,72.7,12.5,7.4,8.3,17.8,49.6,30.8,...,70.6,76.5,22.2,23.2,23.8,26.6,7.2,77.8,54.6,42.1
4,Mar Vista,47.3,27.5,51.3,40.4,35.9,33.6,31.7,32.5,18.9,...,38.7,35.9,36.2,36.2,53.0,52.3,34.7,37.2,28.2,70.0


In [7]:
# set neighborhoods as index in community expense and store distances
community_sales.set_index('neighborhoods', inplace=True)
store_distances.set_index('neighborhoods', inplace=True)

In [8]:
# inspect store preferences
store_preferences.head()

,store,avg_travel_time,business_communities,highways,design,accessibility,parking_space,store_size
0,"Walmart Supercenter, 8500 Washington Blvd, Pic...",48.8,1,4,27,17,1,21658.90
1,"Walmart, 8333 Van Nuys Blvd, Panorama City",24.4,4,3,39,128,1,6846.67
2,"Walmart Supercenter, 7250 Carson Blvd, Long Beach",49.2,0,6,45,64,1,14722.62
3,"Walmart Supercenter, 19821 Rinaldi St, Porter ...",26.3,0,6,27,141,1,14264.98
4,"Target, 8840 Corbin Ave, Northridge",29.8,3,7,52,235,1,10838.32


In [9]:
# set store as index
store_preferences.set_index('store', inplace=True)

#### Attractiveness Measure

In [10]:
# instantiate min-max scaler
scaler = MinMaxScaler()

In [11]:
# scale store preferences between 0 and 1
store_preferences_scaled = scaler.fit_transform(store_preferences)
store_preferences_scaled = pd.DataFrame(store_preferences_scaled, index=store_preferences.index,
                                        columns=store_preferences.columns)

In [12]:
# measure attractiveness
attractiveness = store_preferences.sum(axis=1)

#### Numerator

In [13]:
# create the dataframe that serves as the numerator in the Huff model
numerator_df = pd.DataFrame([], index=store_distances.index)

In [14]:
# calculate the numerator
for col in store_distances.columns:
    numerator_df[col] = attractiveness[col] / (store_distances[col])**2

#### Denominator

In [15]:
# calculate the denominator
denominator = numerator_df.sum(axis=1)

#### Probability of visit

In [16]:
# create the probability dataframe
prob_df = pd.DataFrame([], index=store_distances.index)

In [17]:
for col in numerator_df.columns:
    prob_df[col] = numerator_df[col]/denominator

#### Probable Expense

In [18]:
# create the probable sales dataframe
sales_df = pd.DataFrame([], index=store_distances.index)

In [19]:
# calculate the probable sales of each neighborhood in each store
for col in prob_df.columns:
    sales_df[col] = prob_df[col] * community_sales['qtrly_sales']

In [20]:
# sum up the probable customer sales at each store location
sales = round(sales_df.sum(axis=0), 2).to_dict()

In [21]:
# select the store with the max probable sales
max_sales = max(sales.values())
optimum_stores = [key for key in sales.keys() if sales[key] == max_sales]
# print result
if len(optimum_stores) == 1:
    print(f'''The optimum store location is {optimum_stores[0]} with probable quarterly sales of ${max_sales:,}''')
else:
    optimum_stores = "\n".join(optimum_stores)
    print(f'''The optimum store locations with probable sales of ${max_sales:,} are:\n\n{optimum_stores}''')

The optimum store location is Target, 3535 S La Cienega Blvd, Los Angeles with probable quarterly sales of $354,978.93
